# D06. Итеративный структурный прунинг ResNet-20 на CIFAR-10

как физическое удаление внутренних каналов остаточных блоков влияет на качество и стоимость инференса. исходная модель, три шага прунинга и плотный контроль с тем же бюджетом дообучения. 

Протокол и теория описаны в [отчёте](description_res/description_d6.md).

In [2]:
import copy
import csv
import json
import math
import platform
import random
import time
from datetime import datetime, timezone
from io import BytesIO
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pyarrow.parquet as pq
import torch
from torch import nn
from torch.nn import functional as F
from PIL import Image

ROOT = Path.cwd()
if (ROOT / "pruning").is_dir():
    ROOT = ROOT / "pruning"
assert (ROOT / "description_res").is_dir(), "Запустите ноутбук из корня проекта или pruning/"

SEED = 42
BASE_EPOCHS = 30
RECOVERY_EPOCHS = 3
TARGETS = (0.125, 0.25, 0.50)
BATCH = 256
DATA_REVISION = "0b2714987fa478483af9968de7c934580d0bb9a2"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.allow_tf32 = True
torch.set_num_threads(min(8, torch.get_num_threads()))

log_dir = ROOT / "logs" / "d06" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
log_dir.mkdir(parents=True)
config = {
    "seed": SEED, "architecture": "ResNet-20 (projection shortcuts)", "dataset": "CIFAR-10",
    "dataset_revision": DATA_REVISION,
    "train_count": 45000, "validation_count": 5000, "test_count": 10000,
    "base_epochs": BASE_EPOCHS, "recovery_epochs_per_step": RECOVERY_EPOCHS,
    "targets": TARGETS, "batch_size": BATCH, "optimizer": "SGD",
    "base_start_lr": 0.1, "recovery_start_lr": 0.025,
    "momentum": 0.9, "weight_decay": 0.0005, "dtype": "float32",
    "cudnn_deterministic": True, "cudnn_tf32": True,
    "device": torch.cuda.get_device_name() if DEVICE.type == "cuda" else platform.processor(),
    "torch": torch.__version__, "cuda": torch.version.cuda, "platform": platform.platform(),
    "benchmark_warmup": 25, "benchmark_repeats": 5, "benchmark_forwards_per_repeat": 100,
}
(log_dir / "config.json").write_text(json.dumps(config, indent=2) + "\n")
print("Device:", DEVICE, config["device"])
print("Logs:", log_dir.relative_to(ROOT))

Device: cuda NVIDIA GeForce RTX 4080 SUPER
Logs: logs/d06/20261120T123453299134Z


In [2]:
mean = torch.tensor([0.4914, 0.4822, 0.4465], device=DEVICE)[None, :, None, None]
std = torch.tensor([0.2470, 0.2435, 0.2616], device=DEVICE)[None, :, None, None]

def load_cifar(split, expected_count):
    data_dir = ROOT / "data"
    data_dir.mkdir(exist_ok=True)
    path = data_dir / f"cifar10-{split}-{DATA_REVISION}.parquet"
    if not path.exists():
        url = (f"https://huggingface.co/datasets/uoft-cs/cifar10/resolve/{DATA_REVISION}"
               f"/plain_text/{split}-00000-of-00001.parquet")
        partial = path.with_suffix(".part")
        urlretrieve(url, partial)
        partial.replace(path)
    table = pq.read_table(path, columns=["img", "label"])
    assert len(table) == expected_count
    images = np.empty((expected_count, 32, 32, 3), dtype=np.uint8)
    for index, encoded in enumerate(table["img"].to_pylist()):
        with Image.open(BytesIO(encoded["bytes"])) as image:
            images[index] = image.convert("RGB")
    x = torch.from_numpy(images).to(DEVICE).permute(0, 3, 1, 2).float()
    y = torch.tensor(table["label"].to_pylist(), device=DEVICE)
    return (x / 255 - mean) / std, y

images, labels = load_cifar("train", 50000)
split = torch.randperm(len(labels), generator=torch.Generator().manual_seed(SEED)).to(DEVICE)
train_x, train_y = images[split[5000:]], labels[split[5000:]]
val_x, val_y = images[split[:5000]], labels[split[:5000]]
test_x, test_y = load_cifar("test", 10000)
del images, labels
padded = F.pad(train_x, (4, 4, 4, 4), mode="reflect").permute(0, 2, 3, 1)
pixels = torch.arange(32, device=DEVICE)

def augment(ids, generator):
    offsets = torch.randint(9, (2, len(ids), 1, 1), device=DEVICE, generator=generator)
    flip = torch.rand((len(ids), 1, 1), device=DEVICE, generator=generator) < 0.5
    rows = offsets[0] + pixels[None, :, None]
    cols = offsets[1] + torch.where(flip, 31 - pixels, pixels)
    return padded[ids[:, None, None], rows, cols].permute(0, 3, 1, 2)

У каждого residual-блока отбираются выходные фильтры первой свёртки по сумме модулей весов. Удаляем те же признаки из BatchNorm и входа второй свёртки. Выход второй свёртки и shortcut сохраняют одинаковую ширину, поэтому сложение остаётся корректным. Срезанные тензоры становятся весами новых компактных слоёв.

In [3]:
class Block(nn.Module):
    def __init__(self, in_channels, channels, stride):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)
        self.shortcut = nn.Identity() if stride == 1 else nn.Sequential(
            nn.Conv2d(in_channels, channels, 1, stride, bias=False),
            nn.BatchNorm2d(channels),
        )

    def forward(self, x):
        hidden = F.relu(self.bn1(self.conv1(x)))
        return F.relu(self.bn2(self.conv2(hidden)) + self.shortcut(x))


def resnet20():
    layers = [nn.Conv2d(3, 16, 3, padding=1, bias=False), nn.BatchNorm2d(16), nn.ReLU()]
    previous = 16
    for width in (16, 32, 64):
        for _ in range(3):
            layers.append(Block(previous, width, 1 if previous == width else 2))
            previous = width
    layers.extend([nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, 10)])
    model = nn.Sequential(*layers)
    for layer in model.modules():
        if isinstance(layer, nn.Conv2d):
            nn.init.kaiming_normal_(layer.weight, mode="fan_out", nonlinearity="relu")
    return model.to(device=DEVICE, memory_format=torch.channels_last)


@torch.no_grad()
def prune_channels(model, target):
    for block in (layer for layer in model if isinstance(layer, Block)):
        first, norm, second = block.conv1, block.bn1, block.conv2
        keep_count = round(second.out_channels * (1 - target))
        score = first.weight.abs().sum(dim=(1, 2, 3))
        keep = score.argsort(descending=True, stable=True)[:keep_count].sort().values
        new_first = nn.Conv2d(first.in_channels, keep_count, 3, first.stride, 1, bias=False)
        new_norm = nn.BatchNorm2d(keep_count)
        new_second = nn.Conv2d(keep_count, second.out_channels, 3, padding=1, bias=False)
        new_first.to(device=DEVICE, memory_format=torch.channels_last)
        new_norm.to(DEVICE)
        new_second.to(device=DEVICE, memory_format=torch.channels_last)
        new_first.weight.copy_(first.weight[keep])
        new_second.weight.copy_(second.weight[:, keep])
        for name in ("weight", "bias", "running_mean", "running_var"):
            getattr(new_norm, name).copy_(getattr(norm, name)[keep])
        new_norm.num_batches_tracked.copy_(norm.num_batches_tracked)
        block.conv1, block.bn1, block.conv2 = new_first, new_norm, new_second
        assert block.conv1.out_channels == block.bn1.num_features == block.conv2.in_channels == keep_count

In [4]:
@torch.inference_mode()
def evaluate(model, x, y):
    model.eval()
    loss = torch.zeros((), device=DEVICE)
    correct = torch.zeros((), device=DEVICE, dtype=torch.long)
    for inputs, targets in zip(x.split(BATCH), y.split(BATCH)):
        logits = model(inputs.contiguous(memory_format=torch.channels_last))
        loss += F.cross_entropy(logits, targets, reduction="sum")
        correct += (logits.argmax(1) == targets).sum()
    return float(loss / len(y)), 100 * int(correct) / len(y)


def log_row(filename, row):
    path = log_dir / filename
    has_header = path.exists()
    with path.open("a", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=list(row))
        if not has_header:
            writer.writeheader()
        writer.writerow(row)


def fit(model, branch, step, epochs, start_lr):
    optimizer = torch.optim.SGD(model.parameters(), lr=start_lr, momentum=0.9, weight_decay=5e-4)
    generator = torch.Generator(device=DEVICE).manual_seed(SEED + step)
    for epoch in range(epochs):
        started = time.perf_counter()
        model.train()
        lr = start_lr * (1 + math.cos(math.pi * epoch / epochs)) / 2
        optimizer.param_groups[0]["lr"] = lr
        loss_sum = torch.zeros((), device=DEVICE)
        for ids in torch.randperm(len(train_x), device=DEVICE, generator=generator).split(BATCH):
            optimizer.zero_grad(set_to_none=True)
            logits = model(augment(ids, generator))
            loss = F.cross_entropy(logits, train_y[ids])
            loss.backward()
            optimizer.step()
            loss_sum += loss.detach() * len(ids)
        train_loss = float(loss_sum / len(train_x))
        train_s = time.perf_counter() - started
        val_loss, val_acc = evaluate(model, val_x, val_y)
        log_row("epochs.csv", dict(branch=branch, step=step, epoch=epoch + 1, lr=lr,
                                   train_loss=train_loss, val_loss=val_loss, val_acc_pct=val_acc,
                                   train_s=train_s))
        print(f"{branch:6} step {step} epoch {epoch + 1:2}/{epochs}: "
              f"val={val_acc:.2f}% train={train_s:.2f}s", flush=True)


@torch.inference_mode()
def benchmark(model, batch_size):
    x = test_x[:batch_size].contiguous(memory_format=torch.channels_last)
    for _ in range(config["benchmark_warmup"]):
        model(x)
    samples = []
    for _ in range(config["benchmark_repeats"]):
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        start = time.perf_counter()
        for _ in range(config["benchmark_forwards_per_repeat"]):
            model(x)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        samples.append(1000 * (time.perf_counter() - start) / config["benchmark_forwards_per_repeat"])
    return sorted(samples)[len(samples) // 2]


@torch.inference_mode()
def measure(model, branch, step, phase, target):
    val_loss, val_acc = evaluate(model, val_x, val_y)
    test_loss, test_acc = evaluate(model, test_x, test_y)
    macs = []

    def count_macs(layer, inputs, output):
        if isinstance(layer, nn.Conv2d):
            macs.append(output.numel() * (layer.in_channels // layer.groups) * math.prod(layer.kernel_size))
        else:
            macs.append(output.numel() * layer.in_features)

    hooks = [layer.register_forward_hook(count_macs) for layer in model.modules()
             if isinstance(layer, (nn.Conv2d, nn.Linear))]
    output = model(test_x[:1].contiguous(memory_format=torch.channels_last))
    for hook in hooks:
        hook.remove()
    assert output.shape == (1, 10) and torch.isfinite(output).all()
    row = dict(branch=branch, step=step, phase=phase, target_channel_removal=target,
               params=sum(p.numel() for p in model.parameters()), macs=sum(macs),
               val_loss=val_loss, val_acc_pct=val_acc, test_loss=test_loss, test_acc_pct=test_acc,
               latency_b1_ms=benchmark(model, 1), latency_b128_ms=benchmark(model, 128))
    log_row("metrics.csv", row)
    print(f"{branch:6} step {step} {phase}: {test_acc:.2f}% test, "
          f"{row['params']:,} params, {row['macs']:,} MAC", flush=True)
    return row

In [5]:
base = resnet20()
fit(base, "base", 0, BASE_EPOCHS, config["base_start_lr"])
baseline = measure(base, "base", 0, "trained", 0)
pruned, dense = copy.deepcopy(base), copy.deepcopy(base)
assert sum(isinstance(layer, Block) for layer in base) == 9
results = [baseline]
last_params, last_macs = baseline["params"], baseline["macs"]

for step, target in enumerate(TARGETS, 1):
    prune_channels(pruned, target)
    immediate = measure(pruned, "pruned", step, "before_recovery", target)
    assert immediate["params"] < last_params and immediate["macs"] < last_macs
    last_params, last_macs = immediate["params"], immediate["macs"]
    results.append(immediate)
    fit(pruned, "pruned", step, RECOVERY_EPOCHS, config["recovery_start_lr"])
    recovered = measure(pruned, "pruned", step, "after_recovery", target)
    assert (recovered["params"], recovered["macs"]) == (last_params, last_macs)
    results.append(recovered)

    fit(dense, "dense", step, RECOVERY_EPOCHS, config["recovery_start_lr"])
    control = measure(dense, "dense", step, "matched_budget", 0)
    assert (control["params"], control["macs"]) == (baseline["params"], baseline["macs"])
    results.append(control)

base   step 0 epoch  1/30: val=41.48% train=7.57s
base   step 0 epoch  2/30: val=60.14% train=1.78s
base   step 0 epoch  3/30: val=65.20% train=1.84s
base   step 0 epoch  4/30: val=64.54% train=1.78s
base   step 0 epoch  5/30: val=67.54% train=1.80s
base   step 0 epoch  6/30: val=74.38% train=1.74s
base   step 0 epoch  7/30: val=71.92% train=1.77s
base   step 0 epoch  8/30: val=76.30% train=1.85s
base   step 0 epoch  9/30: val=78.06% train=1.82s
base   step 0 epoch 10/30: val=78.84% train=1.84s
base   step 0 epoch 11/30: val=70.42% train=1.81s
base   step 0 epoch 12/30: val=79.16% train=1.83s
base   step 0 epoch 13/30: val=78.60% train=1.76s
base   step 0 epoch 14/30: val=83.84% train=1.85s
base   step 0 epoch 15/30: val=83.24% train=1.88s
base   step 0 epoch 16/30: val=81.92% train=1.87s
base   step 0 epoch 17/30: val=81.84% train=1.87s
base   step 0 epoch 18/30: val=83.66% train=1.88s
base   step 0 epoch 19/30: val=84.30% train=1.84s
base   step 0 epoch 20/30: val=85.94% train=1.86s


In [1]:
steps = list(range(len(TARGETS) + 1))
before = [baseline] + [r for r in results if r["phase"] == "before_recovery"]
after = [baseline] + [r for r in results if r["phase"] == "after_recovery"]
control = [baseline] + [r for r in results if r["branch"] == "dense"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4), layout="constrained")
for rows, label in [(before, "Сразу после"), (after, "После дообучения"), (control, "Плотный контроль")]:
    axes[0].plot(steps, [r["test_acc_pct"] for r in rows], "o-", label=label)
axes[0].set_ylabel("Test accuracy, %")
for metric, label in [("params", "Параметры"), ("macs", "MAC на изображение")]:
    axes[1].plot(steps, [100 * r[metric] / baseline[metric] for r in after], "o-", label=label)
axes[1].set_ylabel("Доля от исходной модели, %")
for metric, label in [("latency_b1_ms", "Batch 1"), ("latency_b128_ms", "Batch 128")]:
    axes[2].plot(steps, [100 * r[metric] / baseline[metric] for r in after], "o-", label=label)
axes[2].axhline(100, color="gray", linestyle=":")
axes[2].set_ylabel("Задержка, % от исходной модели")
for ax in axes:
    ax.set_xticks(steps)
    ax.set_xlabel("Шаг прунинга")
    ax.legend()
    ax.grid(alpha=0.25)
plt.show()

NameError: name 'TARGETS' is not defined